# 02. Gradient Descent

**Nivel:** 🟢 | **Tiempo:** 75-90 min | **Prereq:** Notebook 01, Cálculo

## 📋 Tabla de Contenidos

1. [Motivación](#motivacion)
2. [Intuición Visual](#intuicion)
3. [Fundamentos Matemáticos](#fundamentos)
4. [Implementación](#implementacion)
5. [**🎯 Ejercicios GRADED**](#graded) ⭐
6. [Framework](#framework)
7. [Papers](#papers)
8. [Best Practices](#practices)
9. [Conclusiones](#conclusiones)
10. [Navegación](#nav)

## 🎯 Objetivos

- Dominar el algoritmo de optimización fundamental del ML
- Implementar: Batch GD, SGD, Mini-batch GD
- Entender learning rate, convergencia, momentum
- Aplicar optimizadores avanzados (Adam, RMSprop)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
import seaborn as sns
plt.style.use('seaborn-v0_8-darkgrid')
np.random.seed(42)
print('✅ Imports OK')

---
## 1. 💡 Motivación

### Por qué Gradient Descent es EL algoritmo de ML

**Gradient Descent es el corazón de:**
- 🔥 Entrenar TODAS las redes neuronales (GPT-4 tiene 1.7T parámetros)
- 🎯 Optimizar cualquier función diferenciable
- 💰 Valor estimado: Trillones de dólares en aplicaciones AI

### 🌍 Casos de Uso Reales

**1. Deep Learning @ Google/Meta/OpenAI** 🧠
- **Input:** Gradientes de billones de parámetros
- **Output:** ChatGPT, DALL-E, modelos de traducción
- **Optimizador:** Adam (variante de GD)
- **Impacto:** Revolución de IA generativa ($100B+ industria)

**2. Recomendaciones @ Netflix/Spotify** 🎬
- **Input:** Matriz 500M usuarios × 100K items
- **Output:** Predicciones personalizadas
- **Método:** SGD con embedding optimization
- **Impacto:** 75% del contenido visto viene de recomendaciones

**3. Trading @ Jane Street/Citadel** 💹
- **Input:** Features de mercado en tiempo real
- **Output:** Señales de trading
- **Método:** Online gradient descent
- **Impacto:** Millisecond-level optimization, billions in profit

**4. Robótica @ Boston Dynamics** 🤖
- **Input:** Estados del robot (posición, velocidad)
- **Output:** Políticas de control óptimas
- **Método:** Policy gradient descent
- **Impacto:** Robots que caminan, saltan, bailan

### 💼 Por qué es fundamental

| Aspecto | Descripción |
|---------|-------------|
| **Universalidad** | Funciona para cualquier función diferenciable |
| **Escalabilidad** | De 10 a 175B parámetros (GPT-3) |
| **Simplicidad** | Concepto simple: seguir la pendiente |
| **Flexibilidad** | Múltiples variantes para diferentes casos |

> **"Gradient descent es a ML lo que el motor de combustión es a los autos"**  
> — Yann LeCun, pionero de Deep Learning

---
## 2. 📊 Intuición Visual

### La Metáfora de la Montaña

Imagina que estás en una montaña con niebla densa y quieres bajar al valle:

**Estrategia humana:**
1. Sientes la pendiente bajo tus pies (gradiente)
2. Das un paso en dirección opuesta a la pendiente
3. Repites hasta llegar abajo

**Estrategia matemática:**
$$\theta_{t+1} = \theta_t - \alpha \nabla J(\theta_t)$$

Donde:
- $\theta$: Tu posición (parámetros del modelo)
- $\alpha$: Tamaño del paso (learning rate)
- $\nabla J$: Pendiente (gradiente de la función de pérdida)

In [ ]:
# Visualización 1: Superficie 3D con GD
fig = plt.figure(figsize=(16, 6))

# Función bowl simple: f(x,y) = x² + y²
x = np.linspace(-5, 5, 100)
y = np.linspace(-5, 5, 100)
X, Y = np.meshgrid(x, y)
Z = X**2 + Y**2

# Gradient descent desde (4, 4)
path = []
theta = np.array([4.0, 4.0])
lr = 0.1
for i in range(50):
    grad = 2 * theta  # Gradiente de x² + y²
    theta = theta - lr * grad
    path.append(theta.copy())
path = np.array(path)

# Plot 1: 3D surface
ax1 = fig.add_subplot(121, projection='3d')
ax1.plot_surface(X, Y, Z, alpha=0.3, cmap='viridis')
ax1.plot(path[:, 0], path[:, 1], path[:, 0]**2 + path[:, 1]**2, 
         'r.-', linewidth=2, markersize=8, label='GD path')
ax1.set_xlabel('θ₁', fontsize=12)
ax1.set_ylabel('θ₂', fontsize=12)
ax1.set_zlabel('Loss', fontsize=12)
ax1.set_title('Gradient Descent en 3D', fontsize=14, fontweight='bold')
ax1.legend()

# Plot 2: Contour con path
ax2 = fig.add_subplot(122)
contour = ax2.contour(X, Y, Z, levels=20, cmap='viridis')
ax2.clabel(contour, inline=True, fontsize=8)
ax2.plot(path[:, 0], path[:, 1], 'r.-', linewidth=2, markersize=8, label='GD path')
ax2.plot(0, 0, 'g*', markersize=20, label='Mínimo global')
ax2.set_xlabel('θ₁', fontsize=12, fontweight='bold')
ax2.set_ylabel('θ₂', fontsize=12, fontweight='bold')
ax2.set_title('Contorno: Descenso al Mínimo', fontsize=14, fontweight='bold')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f'Inicio: {[4.0, 4.0]} → Final: [{path[-1][0]:.6f}, {path[-1][1]:.6f}]')
print(f'Loss inicial: {4**2 + 4**2:.2f} → Loss final: {path[-1][0]**2 + path[-1][1]**2:.6f}')
print('✅ ¡Converged al mínimo!')

In [ ]:
# Visualización 2: Efecto del Learning Rate
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

learning_rates = [0.01, 0.1, 0.9]  # Bajo, bueno, alto
colors = ['blue', 'green', 'red']

for idx, (lr, color) in enumerate(zip(learning_rates, colors)):
    ax = axes[idx]
    
    # GD con diferente lr
    theta = np.array([4.0, 4.0])
    path = [theta.copy()]
    for i in range(50):
        grad = 2 * theta
        theta = theta - lr * grad
        path.append(theta.copy())
    path = np.array(path)
    
    # Plot
    contour = ax.contour(X, Y, Z, levels=20, cmap='gray', alpha=0.3)
    ax.plot(path[:, 0], path[:, 1], f'{color}.-', linewidth=2, markersize=6)
    ax.plot(0, 0, 'g*', markersize=20)
    ax.set_xlabel('θ₁', fontsize=11)
    ax.set_ylabel('θ₂', fontsize=11)
    ax.set_title(f'Learning Rate = {lr}', fontsize=12, fontweight='bold')
    ax.grid(True, alpha=0.3)
    
    # Anotar comportamiento
    if lr == 0.01:
        ax.text(0.05, 0.95, 'Muy lento\n~50 iters', transform=ax.transAxes,
                bbox=dict(boxstyle='round', facecolor='lightblue'),
                verticalalignment='top', fontsize=10)
    elif lr == 0.1:
        ax.text(0.05, 0.95, 'Óptimo\n~20 iters', transform=ax.transAxes,
                bbox=dict(boxstyle='round', facecolor='lightgreen'),
                verticalalignment='top', fontsize=10)
    else:
        ax.text(0.05, 0.95, 'Oscilatorio\nNo converge bien', transform=ax.transAxes,
                bbox=dict(boxstyle='round', facecolor='salmon'),
                verticalalignment='top', fontsize=10)

plt.tight_layout()
plt.show()

print('💡 Learning rate demasiado bajo → Converge lento')
print('💡 Learning rate óptimo → Converge rápido y suave')
print('💡 Learning rate demasiado alto → Oscilaciones, puede divergir')

---
## 3. 📐 Fundamentos Matemáticos

### 3.1 Derivación del Algoritmo

**Objetivo:** Minimizar $J(\theta)$ (función de pérdida)

**Aproximación de Taylor (primer orden):**
$$J(\theta + \Delta\theta) \approx J(\theta) + \nabla J(\theta)^T \Delta\theta$$

Para que $J$ disminuya, necesitamos:
$$\nabla J(\theta)^T \Delta\theta < 0$$

**Solución:** Moverse en dirección opuesta al gradiente
$$\Delta\theta = -\alpha \nabla J(\theta)$$

Por lo tanto:
$$\theta_{t+1} = \theta_t - \alpha \nabla J(\theta_t)$$

### 3.2 Variantes de Gradient Descent

#### Batch Gradient Descent

Usa **todo el dataset** para calcular gradiente:

$$\nabla J(\theta) = \frac{1}{n}\sum_{i=1}^{n} \nabla J_i(\theta)$$

**Pros:**
- ✅ Converge al mínimo exacto (funciones convexas)
- ✅ Actualizaciones estables

**Cons:**
- ❌ Lento para datasets grandes (n > 1M)
- ❌ Mucha memoria (carga todo en RAM)

#### Stochastic Gradient Descent (SGD)

Usa **una muestra** a la vez:

$$\theta_{t+1} = \theta_t - \alpha \nabla J_i(\theta_t)$$

**Pros:**
- ✅ Muy rápido (actualiza en cada muestra)
- ✅ Puede escapar mínimos locales (ruido)
- ✅ Poca memoria

**Cons:**
- ❌ Ruidoso, nunca converge exactamente
- ❌ Requiere decay del learning rate

#### Mini-batch Gradient Descent

Balance: usa **batch pequeño** (32-256):

$$\nabla J(\theta) = \frac{1}{b}\sum_{i \in \text{batch}} \nabla J_i(\theta)$$

**Pros:**
- ✅ Balance velocidad/estabilidad
- ✅ Aprovecha vectorización (GPU)
- ✅ Menos ruidoso que SGD

**Cons:**
- ❌ Hiperparámetro adicional (batch size)

### 3.3 Condiciones de Convergencia

Para SGD, bajo ciertas condiciones del learning rate $\alpha_t$:

1. $\sum_{t=1}^{\infty} \alpha_t = \infty$ (suma infinita)
2. $\sum_{t=1}^{\infty} \alpha_t^2 < \infty$ (suma de cuadrados finita)

**Ejemplo:** $\alpha_t = \frac{\alpha_0}{1 + \beta t}$ satisface ambas

### 3.4 Momentum

Acelera convergencia usando historial:

$$v_t = \gamma v_{t-1} + \alpha \nabla J(\theta_t)$$
$$\theta_{t+1} = \theta_t - v_t$$

Donde $\gamma \in [0.9, 0.99]$ típicamente.

**Intuición:** Una bola rodando acumula velocidad

---
## 4. 💻 Implementación Desde Cero

Implementaremos las 3 variantes principales.

In [ ]:
class GradientDescentOptimizer:
    """
    Gradient Descent con múltiples variantes.
    
    Soporta: batch, sgd, mini-batch, momentum
    """
    
    def __init__(self, method='mini-batch', learning_rate=0.01, 
                 batch_size=32, momentum=0.0, n_iterations=1000):
        """
        Parameters:
        -----------
        method : str
            'batch', 'sgd', o 'mini-batch'
        learning_rate : float
            Tamaño del paso
        batch_size : int
            Tamaño del batch para mini-batch
        momentum : float
            Factor de momentum [0, 1]
        n_iterations : int
            Número de iteraciones/épocas
        """
        self.method = method
        self.lr = learning_rate
        self.batch_size = batch_size
        self.momentum = momentum
        self.n_iters = n_iterations
        self.losses = []
        self.w = None
        self.b = None
        self.velocity_w = None
        self.velocity_b = None
        
    def fit(self, X, y):
        """Entrenar con el método seleccionado"""
        n_samples, n_features = X.shape
        
        # Inicializar parámetros
        self.w = np.zeros(n_features)
        self.b = 0
        
        # Inicializar momentum
        if self.momentum > 0:
            self.velocity_w = np.zeros(n_features)
            self.velocity_b = 0
        
        # Entrenar según método
        if self.method == 'batch':
            self._batch_gd(X, y)
        elif self.method == 'sgd':
            self._stochastic_gd(X, y, n_samples)
        else:  # mini-batch
            self._minibatch_gd(X, y, n_samples)
        
        return self
    
    def _batch_gd(self, X, y):
        """Batch Gradient Descent"""
        for epoch in range(self.n_iters):
            # Calcular gradiente en todo el dataset
            dw, db = self._compute_gradients(X, y)
            
            # Actualizar con momentum si aplica
            if self.momentum > 0:
                self.velocity_w = self.momentum * self.velocity_w + self.lr * dw
                self.velocity_b = self.momentum * self.velocity_b + self.lr * db
                self.w -= self.velocity_w
                self.b -= self.velocity_b
            else:
                self.w -= self.lr * dw
                self.b -= self.lr * db
            
            # Guardar loss
            loss = self._compute_loss(X, y)
            self.losses.append(loss)
            
            if (epoch + 1) % 100 == 0:
                print(f'Epoch {epoch+1}/{self.n_iters}, Loss: {loss:.6f}')
    
    def _stochastic_gd(self, X, y, n_samples):
        """Stochastic Gradient Descent"""
        for epoch in range(self.n_iters):
            # Shuffle data
            indices = np.random.permutation(n_samples)
            
            for i in indices:
                # Gradiente de UNA muestra
                Xi = X[i:i+1]
                yi = y[i:i+1]
                dw, db = self._compute_gradients(Xi, yi)
                
                # Actualizar
                if self.momentum > 0:
                    self.velocity_w = self.momentum * self.velocity_w + self.lr * dw
                    self.velocity_b = self.momentum * self.velocity_b + self.lr * db
                    self.w -= self.velocity_w
                    self.b -= self.velocity_b
                else:
                    self.w -= self.lr * dw
                    self.b -= self.lr * db
            
            # Loss al final de época
            loss = self._compute_loss(X, y)
            self.losses.append(loss)
    
    def _minibatch_gd(self, X, y, n_samples):
        """Mini-batch Gradient Descent"""
        for epoch in range(self.n_iters):
            indices = np.random.permutation(n_samples)
            
            for start in range(0, n_samples, self.batch_size):
                end = min(start + self.batch_size, n_samples)
                batch_idx = indices[start:end]
                
                # Gradiente del batch
                dw, db = self._compute_gradients(X[batch_idx], y[batch_idx])
                
                # Actualizar
                if self.momentum > 0:
                    self.velocity_w = self.momentum * self.velocity_w + self.lr * dw
                    self.velocity_b = self.momentum * self.velocity_b + self.lr * db
                    self.w -= self.velocity_w
                    self.b -= self.velocity_b
                else:
                    self.w -= self.lr * dw
                    self.b -= self.lr * db
            
            # Loss al final de época
            loss = self._compute_loss(X, y)
            self.losses.append(loss)
    
    def _compute_gradients(self, X, y):
        """Calcular gradientes"""
        n = len(y)
        y_pred = X @ self.w + self.b
        error = y_pred - y
        dw = (2/n) * (X.T @ error)
        db = (2/n) * np.sum(error)
        return dw, db
    
    def _compute_loss(self, X, y):
        """Calcular MSE loss"""
        y_pred = X @ self.w + self.b
        return np.mean((y - y_pred) ** 2)
    
    def predict(self, X):
        """Predicciones"""
        return X @ self.w + self.b

print('✅ Clase GradientDescentOptimizer implementada')
print('   Métodos: batch, sgd, mini-batch')
print('   Features: momentum, loss tracking')

In [ ]:
# DEMO 1: Comparar los 3 métodos
print('='*70)
print('DEMO: COMPARACIÓN DE MÉTODOS')
print('='*70)

# Datos sintéticos
np.random.seed(42)
X_train = np.random.randn(1000, 5)
true_w = np.array([3, -2, 1, 4, -1])
y_train = X_train @ true_w + 5 + np.random.randn(1000) * 0.5

# Entrenar con cada método
models = {
    'Batch GD': GradientDescentOptimizer(method='batch', learning_rate=0.01, n_iterations=100),
    'SGD': GradientDescentOptimizer(method='sgd', learning_rate=0.001, n_iterations=20),
    'Mini-batch': GradientDescentOptimizer(method='mini-batch', learning_rate=0.01, 
                                            batch_size=32, n_iterations=100)
}

results = {}
for name, model in models.items():
    print(f'\n--- {name} ---')
    import time
    start = time.time()
    model.fit(X_train, y_train)
    elapsed = time.time() - start
    
    final_loss = model.losses[-1]
    results[name] = {'loss': final_loss, 'time': elapsed, 'model': model}
    print(f'Final loss: {final_loss:.6f}')
    print(f'Time: {elapsed:.4f}s')
    print(f'Weights: {model.w}')

print(f'\n{"="*70}')
print('CONCLUSIÓN:')
print(f'  - Batch GD: Más lento pero preciso')
print(f'  - SGD: Más rápido pero ruidoso')
print(f'  - Mini-batch: Balance óptimo (usado en Deep Learning)')

In [ ]:
# DEMO 2: Visualizar convergencia
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Plot 1: Curvas de loss
ax = axes[0]
for name, data in results.items():
    losses = data['model'].losses
    ax.plot(losses, linewidth=2, label=name, alpha=0.8)

ax.set_xlabel('Iteración/Época', fontsize=12, fontweight='bold')
ax.set_ylabel('MSE Loss', fontsize=12, fontweight='bold')
ax.set_title('Convergencia de Diferentes Métodos', fontsize=14, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
ax.set_yscale('log')

# Plot 2: Predicciones vs True
ax = axes[1]
model_best = results['Mini-batch']['model']
y_pred = model_best.predict(X_train)

ax.scatter(y_train, y_pred, alpha=0.4, s=30, c='steelblue', edgecolors='navy')
ax.plot([y_train.min(), y_train.max()], [y_train.min(), y_train.max()], 
        'r--', linewidth=3, label='Predicción perfecta')
ax.set_xlabel('y real', fontsize=12, fontweight='bold')
ax.set_ylabel('y predicho', fontsize=12, fontweight='bold')
ax.set_title('Mini-batch GD: Resultados', fontsize=14, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)

r2 = 1 - np.sum((y_train - y_pred)**2) / np.sum((y_train - np.mean(y_train))**2)
ax.text(0.05, 0.95, f'R² = {r2:.4f}', transform=ax.transAxes,
        fontsize=13, verticalalignment='top',
        bbox=dict(boxstyle='round', facecolor='lightgreen', alpha=0.9))

plt.tight_layout()
plt.show()

---
## 5. 🎯 Ejercicios GRADED

Total: 100 pts | Mínimo: 70 pts

**Tests automáticos:** `pytest rutas/01-ml-clasico/tests/test_02.py`

### Ejercicio 1: Gradiente Numérico (15 pts)

Implementa cálculo de gradiente por **diferencias finitas**.

**Fórmula:**
$$\frac{\partial f}{\partial x} \approx \frac{f(x + h) - f(x - h)}{2h}$$

Donde $h$ es un valor pequeño (ej: 1e-5).

In [ ]:
def compute_gradient_basic(f, x, h=1e-5):
    """
    Calcular gradiente numérico usando diferencias finitas.
    
    Parameters:
    -----------
    f : function
        Función a derivar (debe aceptar un escalar)
    x : float
        Punto donde calcular derivada
    h : float
        Paso pequeño para diferencias finitas
        
    Returns:
    --------
    gradient : float
        Aproximación de df/dx en x
    """
    # START CODE HERE (≈ 2 líneas)
    # 1. Evalúa f(x + h) y f(x - h)
    # 2. Aplica fórmula de diferencias finitas centrales
    gradient = None
    # END CODE HERE
    return gradient

<details><summary>💡 Hint 1: Evaluación de la función</summary>

Para calcular diferencias finitas centrales necesitas:
1. Evaluar la función en `x + h`: `f_plus = f(x + h)`
2. Evaluar la función en `x - h`: `f_minus = f(x - h)`
3. Calcular la diferencia: `f_plus - f_minus`
4. Dividir por `2*h`
</details>

<details><summary>💡 Hint 2: Fórmula paso a paso</summary>

```python
# Método 1: Paso a paso (más claro)
f_plus = f(x + h)
f_minus = f(x - h)
gradient = (f_plus - f_minus) / (2 * h)

# Método 2: Una línea (más compacto)
# gradient = (f(x + h) - f(x - h)) / (2 * h)
```
</details>

<details><summary>🔑 Solución Completa</summary>

```python
def compute_gradient_basic(f, x, h=1e-5):
    """Calcular gradiente numérico"""
    # Diferencias finitas centrales (más preciso que forward/backward)
    f_plus = f(x + h)
    f_minus = f(x - h)
    gradient = (f_plus - f_minus) / (2 * h)
    return gradient
```

**Explicación:**
- Diferencias finitas centrales son más precisas: $O(h^2)$ vs $O(h)$
- `h=1e-5` es un buen balance entre precisión y errores numéricos
- Para `f(x) = x²`, en `x=3` el gradiente debería ser `≈6.0`
- Para `f(x) = sin(x)`, en `x=0` el gradiente debería ser `≈1.0` (cos(0))
</details>

In [ ]:
# 🧪 Test de compute_gradient_basic
print('Testing compute_gradient_basic...')
print('-' * 60)

# Test 1: f(x) = x² → f'(x) = 2x
f1 = lambda x: x**2
x_test = 3.0
result = compute_gradient_basic(f1, x_test)
expected = 2 * x_test  # Derivada analítica
print(f'Test 1: Gradiente de x² en x={x_test}')
print(f'  Resultado numérico: {result:.6f}')
print(f'  Esperado (2*{x_test}): {expected:.6f}')
print(f'  Error: {abs(result - expected):.2e}')
print(f'  ✅ PASS' if abs(result - expected) < 1e-4 else '  ❌ FAIL')

# Test 2: f(x) = sin(x) → f'(x) = cos(x)
import math
f2 = lambda x: math.sin(x)
x_test2 = 0.5
result2 = compute_gradient_basic(f2, x_test2)
expected2 = math.cos(x_test2)
print(f'\nTest 2: Gradiente de sin(x) en x={x_test2}')
print(f'  Resultado numérico: {result2:.6f}')
print(f'  Esperado (cos({x_test2})): {expected2:.6f}')
print(f'  Error: {abs(result2 - expected2):.2e}')
print(f'  ✅ PASS' if abs(result2 - expected2) < 1e-4 else '  ❌ FAIL')

# Test 3: f(x) = e^x → f'(x) = e^x
f3 = lambda x: math.exp(x)
x_test3 = 1.0
result3 = compute_gradient_basic(f3, x_test3)
expected3 = math.exp(x_test3)
print(f'\nTest 3: Gradiente de e^x en x={x_test3}')
print(f'  Resultado: {result3:.6f}')
print(f'  Esperado: {expected3:.6f}')
print(f'  ✅ PASS' if abs(result3 - expected3) < 1e-4 else '  ❌ FAIL')

print(f'\n{"="*60}')
print('💡 Diferencias finitas centrales son precisas!')
print('   Útiles para verificar gradientes analíticos (gradient checking)')

### Ejercicio 2: Paso de GD (20 pts)

In [ ]:
def gradient_descent_step(x, gradient, learning_rate):
    # START CODE HERE (≈ 1 línea)
    x_new = None
    # END CODE HERE
    return x_new

<details><summary>🔑</summary>

```python
return x - learning_rate * gradient
```
</details>

In [ ]:
# Test
x = np.array([1.0, 2.0])
grad = np.array([0.5, -0.3])
new_x = gradient_descent_step(x, grad, 0.1)
print(f'x: {x} → {new_x}')

### Ejercicio 3: Batch GD (30 pts)

In [ ]:
def batch_gradient_descent(X, y, w, learning_rate, epochs):
    # START CODE HERE
    for epoch in range(epochs):
        # 1. Calcular predicciones
        # 2. Calcular gradiente
        # 3. Actualizar w
        pass
    # END CODE HERE
    return w

<details><summary>🔑</summary>

```python
for epoch in range(epochs):
    y_pred = X @ w
    grad = (2/len(y)) * X.T @ (y_pred - y)
    w -= learning_rate * grad
return w
```
</details>

### Ejercicio 4: SGD (15 pts)

In [ ]:
def stochastic_gradient_descent(X, y, w, learning_rate, epochs):
    # START CODE HERE
    # Loop sobre epochs y muestras individuales
    pass
    # END CODE HERE
    return w

### Ejercicio 5: Comparar (20 pts)

In [ ]:
def compare_optimizers(X, y, methods):
    # START CODE HERE
    # Retorna dict con losses finales
    pass
    # END CODE HERE
    pass

---
## 6. 🔧 Framework

scikit-learn usa GD internamente en `SGDRegressor`:

In [ ]:
from sklearn.linear_model import SGDRegressor

model = SGDRegressor(max_iter=1000, learning_rate='adaptive', 
                     eta0=0.01, random_state=42)
model.fit(X_train, y_train)
print(f'Coef: {model.coef_}')
print(f'Loss: {np.mean((y_train - model.predict(X_train))**2):.6f}')

---
## 7. 📄 Papers

### Robbins & Monro (1951)
*A Stochastic Approximation Method*
- Primera formalización de SGD
- Condiciones de convergencia

### Bottou (2010)
*Large-Scale ML with SGD*
- SGD para big data
- Trade-offs teóricos/prácticos

### Kingma & Ba (2015) - **Adam**
*Adam: A Method for Stochastic Optimization*
- Adaptive learning rates
- Combina momentum + RMSprop
- Estado del arte (2015-hoy)
- Link: https://arxiv.org/abs/1412.6980

### Rumelhart et al. (1986)
*Backpropagation*
- Aplicación de GD a redes neuronales
- Fundamento del Deep Learning

---
## 8. ✅ Best Practices

### Elegir Learning Rate

**Reglas empíricas:**
```python
# Demasiado alto (>1): Diverge
lr_bad = 2.0

# Alto (0.1-1): Oscila
lr_oscillate = 0.5

# Bueno (0.001-0.1): Converge suave
lr_good = 0.01

# Bajo (<0.001): Muy lento
lr_slow = 0.0001
```

**Técnicas:**
```python
# Learning rate decay
lr_t = lr_0 / (1 + decay * epoch)

# Step decay
if epoch % 10 == 0:
    lr *= 0.9

# Cosine annealing (PyTorch)
from torch.optim.lr_scheduler import CosineAnnealingLR
```

### Batch Size

| Tamaño | Pros | Cons | Uso |
|--------|------|------|-----|
| 1-32 | Rápido, regulariza | Ruidoso, GPU underutilized | RNNs |
| 32-256 | **Balance óptimo** | - | **Default** |
| 256+ | Smooth, vectorizado | Requiere más RAM, generaliza peor | ResNets grandes |

**Regla:** Usa el batch size más grande que quepa en GPU/RAM

### Cuándo usar qué

- **Batch GD:** Datasets pequeños (<10K), quieres precisión
- **SGD:** Datasets masivos (>1M), online learning
- **Mini-batch:** **Default** (99% de casos)
- **Adam:** Cuando no quieres tunear lr (adapta automáticamente)

---
## 9. 🎓 Conclusiones

### Conceptos Clave

1. **GD es iterativo:** $\theta = \theta - \alpha \nabla J$
2. **3 variantes:** Batch (todo), SGD (uno), Mini-batch (balance)
3. **Learning rate es crítico:** Determina velocidad y convergencia
4. **Trade-offs:** Velocidad vs precisión vs memoria

### Lo que hace a GD especial

| Ventaja | Descripción |
|---------|-------------|
| 🌍 **Universal** | Funciona para cualquier $J$ diferenciable |
| 📈 **Escalable** | De 10 a 175B parámetros |
| 🎯 **Simple** | Conceptualmente elegante |
| 🔧 **Flexible** | Momentum, Adam, RMSprop |

### Próximos pasos

**Notebook 03: Regresión Logística**
- Clasificación binaria
- GD aplicado a cross-entropy
- Interpretación probabilística

**Optimizadores avanzados (fuera del curso):**
- Adam, AdaGrad, RMSprop
- Learning rate scheduling
- Second-order methods (Newton, L-BFGS)

> **"Si entiendes GD, entiendes el 80% del Deep Learning"**  
> — Jeremy Howard, Fast.ai

---
## 10. 📚 Navegación

### Ruta 1: ML Clásico

1. [01 - Regresión Lineal](01-regresion-lineal.ipynb)
2. **[02 - Gradient Descent](02-gradient-descent.ipynb)** ⭐ Estás aquí
3. [03 - Regresión Logística](03-regresion-logistica.ipynb)
4. [04 - Regresión Softmax](04-regresion-softmax.ipynb)
5. [05 - Árboles de Decisión](05-arboles-decision.ipynb)
6. [06 - Random Forests](06-random-forests.ipynb)
7. [07 - Boosting](07-boosting.ipynb)
8. [08 - SVM](08-svm.ipynb)
9. [09 - K-Means](09-kmeans.ipynb)
10. [10 - PCA](10-pca.ipynb)

---

**[⬅️ 01 Regresión Lineal](01-regresion-lineal.ipynb) | [03 Regresión Logística ➡️](03-regresion-logistica.ipynb)**

**[🏠 Volver al índice](../../README.md)**